In [9]:
import pandas as pd
import numpy as np  
df = pd.read_excel('POM PAID-PRIO_ARNG_workingpython.xlsx',sheet_name='Sum Cost')
#df = pd.read_excel('POM PAID-PRIO_ARNG_workingpython.xlsx',sheet_name='Sum Driver')
df.columns = df.columns.str.strip()  # Removes leading/trailing spaces


In [10]:
from statsmodels.tsa.holtwinters import ExponentialSmoothing

# Select numeric columns (years)
years = ["2019", "2020", "2021", "2022", "2023", "2024"]
forecast_years = [str(year) for year in range(2025, 2032)]  # Forecast 7 years

df_forecast = df.copy()  # Create a copy to store forecasts

# Apply forecasting model row by row
for index, row in df.iterrows():
    try:
                # Convert to numeric time series
        time_series = row[years].astype(float).values

        # Apply log transformation (avoid log(0) issue by adding a small constant)
        log_series = np.log(time_series + 1e-5)

        # Fit Holt-Winters model (Triple Exponential Smoothing)
        model = ExponentialSmoothing(log_series, trend="add", seasonal="add", seasonal_periods=3)
        fit = model.fit()

        # Forecast on log scale
        log_forecast = fit.forecast(steps=len(forecast_years))

        # Reverse log transformation
        forecast_values = np.exp(log_forecast) - 1e-5

        # Store forecasts
        for i, year in enumerate(forecast_years):
            df_forecast.at[index, year] = max(forecast_values[i], 0)  # Ensure no negative values
    
    except Exception as e:
        print(f"Skipping row {index} due to error: {e}")


c:\Users\john.voltz\Anaconda3\lib\site-packages\statsmodels\tsa\holtwinters\model.py:1409: RuntimeWarning: divide by zero encountered in log
  aic = self.nobs * np.log(sse / self.nobs) + k * 2
c:\Users\john.voltz\Anaconda3\lib\site-packages\statsmodels\tsa\holtwinters\model.py:1414: RuntimeWarning: invalid value encountered in double_scalars
  aicc = aic + aicc_penalty
c:\Users\john.voltz\Anaconda3\lib\site-packages\statsmodels\tsa\holtwinters\model.py:1415: RuntimeWarning: divide by zero encountered in log
  bic = self.nobs * np.log(sse / self.nobs) + k * np.log(self.nobs)
c:\Users\john.voltz\Anaconda3\lib\site-packages\statsmodels\tsa\holtwinters\model.py:1409: RuntimeWarning: divide by zero encountered in log
  aic = self.nobs * np.log(sse / self.nobs) + k * 2
c:\Users\john.voltz\Anaconda3\lib\site-packages\statsmodels\tsa\holtwinters\model.py:1414: RuntimeWarning: invalid value encountered in double_scalars
  aicc = aic + aicc_penalty
c:\Users\john.voltz\Anaconda3\lib\site-packages

In [11]:
# Save the forecasted data
export_file_path = "Forecasted_HWCost.xlsx"
df_forecast.to_excel(export_file_path, index=False)
print(f"Forecasted data saved as {export_file_path}")

Forecasted data saved as Forecasted_HWCost.xlsx
